# Set environment

In [1]:
### Basic
import numpy  as np
import scipy  as sc
import pandas as pd
import glob, os, re
import pickle, sys

### import flowImg
py_path = "/local_data/notebooks/clintko/GitRepo/Proj_Flow2Image"
sys.path.insert(0, py_path)
from flowImg import *

### global const
SEED    = 123
DAT_DIR = "/data/clintko/flow_EQAPOL_normal/pipe01"
RES_DIR = "/data/clintko/flow_EQAPOL_normal/mpsk"

/local_data/env-py3/lib/python3.6/site-packages/h5py/__init__.py:36: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
Using TensorFlow backend.


# Load data

In [2]:
#!mkdir /data/clintko/flow_EQAPOL_normal/mpsk
!ls /data/clintko/flow_EQAPOL_normal

C_1e4.txt	       data_costim_1e3.npy  data_seb_1e5.npy  mpsk
data_cmv_pp65_1e3.npy  data_costim_1e4.npy  data_seb.npy      pipe01
data_cmv_pp65_1e4.npy  data_costim_1e5.npy  markers.csv       Y_cmv_1e4.txt
data_cmv_pp65_1e5.npy  data_costim.npy	    matrix_1e3.npy    Y_costim_1e4.txt
data_cmv_pp65.npy      data_seb_1e3.npy     matrix_1e4.npy    Y_seb_1e4.txt
data_combine.P	       data_seb_1e4.npy     matrix_1e5.npy


In [3]:
!ls /data/clintko/flow_EQAPOL_normal/mpsk

C_1e4.csv  Y_1e4_cmv.csv  Y_1e4_costim.csv  Y_1e4_seb.csv


In [4]:
!ls /data/clintko/flow_EQAPOL_normal/pipe01

wrap01_data.P  wrap02_scaled_k0.P


In [5]:
fname = "wrap02_scaled_k0.P"
fpath = os.path.join(DAT_DIR, fname)
file  = open(fpath, "rb")

wrap_scaled = pickle.load(file)
file.close()

In [6]:
print(wrap_scaled)

Label: Counter({0: 11, 1: 11, 2: 11})
    0: [ 0  1  2  3  4  5  6  7  8  9 10]
    1: [11 12 13 14 15 16 17 18 19 20 21]
    2: [22 23 24 25 26 27 28 29 30 31 32]
------------------
Data Matrix: (15284646, 14)
    #Samples:   33
    #Variables: 14
------------------
Coordinate: None
------------------
Image: None



# Subsample 10,000 for each sample that will be fit into the MPSK model

In [7]:
n_subsample = 1
n_size = 10000

In [8]:
sampler = Transform_Subsampling(
    n_subsample  = n_subsample, 
    n_size       = n_size, 
    random_state = SEED)

wrap_subs_1e4 = sampler.fit_transform(wrap_scaled)

In [9]:
print(wrap_subs_1e4)

Label: Counter({0: 11, 1: 11, 2: 11})
    0: [ 0  1  2  3  4  5  6  7  8  9 10]
    1: [11 12 13 14 15 16 17 18 19 20 21]
    2: [22 23 24 25 26 27 28 29 30 31 32]
------------------
Data Matrix: (330000, 14)
    #Samples:   33
    #Variables: 14
------------------
Coordinate: None
------------------
Image: None



# Get Y and C

In [10]:
### set file name and path
fname = "C_1e4.csv"
fpath = os.path.join(RES_DIR, fname)

### set up the sample label and store
C = np.concatenate([[idx+1] * n_size for idx in range(11)])
C = pd.DataFrame(C)
C.to_csv(fpath, index=False, header=False)

### show the dimension of the label
print(C.shape)

(110000, 1)


In [11]:
print(C.values)

[[ 1]
 [ 1]
 [ 1]
 ...
 [11]
 [11]
 [11]]


In [12]:
label = wrap_subs_1e4.unique_label()
label

array([0, 1, 2])

In [13]:
matrix = wrap_subs_1e4.get_matrix_original()
print(matrix.shape)

(33, 10000, 14)


**costime**

In [14]:
### extract costim samples
idx = wrap_subs_1e4.which_label(label[0])
mat = matrix[idx]
mat = np.concatenate(mat)

### set file name & path
fname = "Y_1e4_costim.csv"
fpath = os.path.join(RES_DIR, fname)

### store the results
Y = pd.DataFrame(mat)
Y.to_csv(fpath, index=False, header=False)

### show the results
print(fpath)
print(idx)
print(mat.shape)

/data/clintko/flow_EQAPOL_normal/mpsk/Y_1e4_costim.csv
[ 0  1  2  3  4  5  6  7  8  9 10]
(110000, 14)


**CMV**

In [15]:
### extract costim samples
idx = wrap_subs_1e4.which_label(label[1])
mat = matrix[idx]
mat = np.concatenate(mat)

### set file name & path
fname = "Y_1e4_cmv.csv"
fpath = os.path.join(RES_DIR, fname)

### store the results
Y = pd.DataFrame(mat)
Y.to_csv(fpath, index=False, header=False)

### show the results
print(fpath)
print(idx)
print(mat.shape)

/data/clintko/flow_EQAPOL_normal/mpsk/Y_1e4_cmv.csv
[11 12 13 14 15 16 17 18 19 20 21]
(110000, 14)


**SEB**

In [16]:
### extract costim samples
idx = wrap_subs_1e4.which_label(label[2])
mat = matrix[idx]
mat = np.concatenate(mat)

### set file name & path
fname = "Y_1e4_seb.csv"
fpath = os.path.join(RES_DIR, fname)

### store the results
Y = pd.DataFrame(mat)
Y.to_csv(fpath, index=False, header=False)

### show the results
print(fpath)
print(idx)
print(mat.shape)

/data/clintko/flow_EQAPOL_normal/mpsk/Y_1e4_seb.csv
[22 23 24 25 26 27 28 29 30 31 32]
(110000, 14)


### check if the file is stored correctly

In [17]:
!ls -l /data/clintko/flow_EQAPOL_normal/mpsk

total 88804
-rw-r--r-- 1 clintko clintko   240000 Mar 23 20:26 C_1e4.csv
-rw-r--r-- 1 clintko clintko 30278794 Mar 23 20:26 Y_1e4_cmv.csv
-rw-r--r-- 1 clintko clintko 30274807 Mar 23 20:26 Y_1e4_costim.csv
-rw-r--r-- 1 clintko clintko 30265195 Mar 23 20:26 Y_1e4_seb.csv
